# Railway Analytics — Lakehouse Exploration

This notebook queries the **Apache Iceberg** table (Silver layer) directly in **MinIO**,
using **DuckDB** — exactly the same strategy as the consumption API.

Prerequisites: the stack is up (`make up`) and the pipeline has already run
(`make simulate` + `make process`).


## 1. DuckDB → MinIO/Iceberg connection


In [ ]:
import os, re, duckdb, boto3

S3_ENDPOINT = os.getenv('S3_ENDPOINT', 'http://minio:9000')
S3_KEY      = os.getenv('S3_ACCESS_KEY', 'minioadmin')
S3_SECRET   = os.getenv('S3_SECRET_KEY', 'minioadmin')
S3_REGION   = os.getenv('S3_REGION', 'us-east-1')
SILVER      = os.getenv('SILVER_BUCKET', 'silver')
WH_PREFIX   = os.getenv('ICEBERG_WAREHOUSE_PREFIX', 'warehouse')
NAMESPACE   = os.getenv('ICEBERG_NAMESPACE', 'railway')
TABLE       = os.getenv('ICEBERG_TABLE', 'train_events')

endpoint_host = S3_ENDPOINT.replace('http://', '').replace('https://', '')

con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute('INSTALL iceberg; LOAD iceberg;')
con.execute(f"SET s3_endpoint='{endpoint_host}';")
con.execute("SET s3_use_ssl=false;")
con.execute("SET s3_url_style='path';")
con.execute(f"SET s3_region='{S3_REGION}';")
con.execute(f"SET s3_access_key_id='{S3_KEY}';")
con.execute(f"SET s3_secret_access_key='{S3_SECRET}';")
con.execute("SET unsafe_enable_version_guessing=true;")
print('DuckDB ready.')


## 2. Locate the most recent Iceberg `metadata.json`

DuckDB reads the table's files from the newest *metadata*.
We list the `metadata/` directory in MinIO and pick the highest version.


In [ ]:
s3 = boto3.client('s3', endpoint_url=S3_ENDPOINT,
                  aws_access_key_id=S3_KEY, aws_secret_access_key=S3_SECRET,
                  region_name=S3_REGION)

prefix = f'{WH_PREFIX}/{NAMESPACE}/{TABLE}/metadata/'
resp = s3.list_objects_v2(Bucket=SILVER, Prefix=prefix)
keys = [o['Key'] for o in resp.get('Contents', []) if o['Key'].endswith('.metadata.json')]

def version(k):
    m = re.search(r'v?(\d+)\.metadata\.json$', k.split('/')[-1])
    return int(m.group(1)) if m else -1

assert keys, f'No metadata in s3://{SILVER}/{prefix} — run the Spark job first.'
latest = max(keys, key=version)
metadata_uri = f's3://{SILVER}/{latest}'
print('metadata:', metadata_uri)

con.execute(f"""
    CREATE OR REPLACE VIEW train_events AS
    SELECT * FROM iceberg_scan('{metadata_uri}')
""")
con.execute('SELECT count(*) AS rows FROM train_events').df()


## 3. Punctuality KPIs

Official rule: delay **> 5 min** ⇒ `DELAYED`. The deduplication by `trip_id`
guarantees **one row per trip** (the Spark UPSERT).


In [ ]:
con.execute("""
    SELECT
        count(*)                                          AS total_trips,
        count(*) FILTER (WHERE delay_status = 'DELAYED')  AS delayed,
        count(*) FILTER (WHERE delay_status = 'ON_TIME')  AS on_time,
        count(*) FILTER (WHERE delay_status = 'UNKNOWN')  AS unknown,
        round(avg(delay_minutes), 2)                      AS avg_delay_min,
        round(max(delay_minutes), 2)                      AS max_delay_min,
        round(100.0 * count(*) FILTER (WHERE delay_status = 'ON_TIME')
              / nullif(count(*), 0), 2)                    AS on_time_pct
    FROM train_events
""").df()


## 4. Worst delays


In [ ]:
con.execute("""
    SELECT trip_id, train_number, operator,
           current_station_name, delay_minutes, delay_status
    FROM train_events
    WHERE delay_minutes IS NOT NULL
    ORDER BY delay_minutes DESC
    LIMIT 10
""").df()


## 5. Average delay per operator (chart)


In [ ]:
import matplotlib.pyplot as plt

df = con.execute("""
    SELECT operator,
           round(avg(delay_minutes), 2) AS avg_delay,
           count(*) AS trips
    FROM train_events
    WHERE operator IS NOT NULL
    GROUP BY operator
    ORDER BY avg_delay DESC
""").df()

ax = df.plot(kind='bar', x='operator', y='avg_delay', legend=False, figsize=(8,4))
ax.set_ylabel('Average delay (min)')
ax.set_xlabel('Operator')
ax.set_title('Average delay per operator')
plt.tight_layout()
plt.show()
df


## 6. Proof of the UPSERT (single record)

Pick any `trip_id`: it appears **exactly once**, with the most recent state
— even though it received several telemetry updates.


In [ ]:
trip = con.execute('SELECT trip_id FROM train_events LIMIT 1').fetchone()[0]
print('trip_id:', trip)
con.execute('SELECT count(*) AS occurrences FROM train_events WHERE trip_id = ?', [trip]).df()
